# 1. Environment & Imports

โหลด autoreload สำหรับให้ Notebook รับโค้ดใหม่จาก `src/` อัตโนมัติเมื่อแก้ไข, import ไลบรารีสำหรับวิเคราะห์/พล็อต (pandas, numpy, matplotlib, seaborn) และ import `TextCleaner` ซึ่งเป็น Pipeline หลักสำหรับทำความสะอาดข้อมูล

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import sys
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.append(os.path.abspath('../src'))

from data_cleaning import TextCleaner

sns.set_theme(style='whitegrid')


# 2. Data Loading & Inspection

โหลดข้อมูลตัวอย่าง (`sample.csv`, ไม่มี header) และตรวจสอบโครงสร้าง/สัดส่วน Missing Values ของข้อมูลดิบก่อนเข้าสู่ขั้นตอนทำความสะอาด

In [4]:
raw_data = pd.read_csv(
    '../data/sample/sample.csv',
    header=None,
    index_col=False,
)
raw_data.head()


,0,1,2
0,1,Expensive Junk,This product consists of a piece of thin flexi...
1,1,Toast too dark,"Even on the lowest setting, the toast is too d..."
2,2,Excellent imagery...dumbed down story,I enjoyed this disc. The video is stunning. I ...
3,1,Are we pretending everyone is married?,The authors pretend that parents neither die n...
4,1,Not worth your time,"Might as well just use a knife, this product h..."


In [5]:
raw_data.info()


<class 'pandas.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   0       50000 non-null  int64
 1   1       49997 non-null  str  
 2   2       50000 non-null  str  
dtypes: int64(1), str(2)
memory usage: 1.1 MB


# 3. Exploratory Data Analysis (EDA)

สำรวจข้อมูล **ก่อน** เข้า Cleaning Pipeline เพื่อดูภาพรวมของ `sample.csv` :



# 4. Data Cleaning Pipeline

รัน Pipeline หลักจาก `TextCleaner` (`src/data_cleaning.py`): rename/map คอลัมน์, แปลง dtype, ล้างข้อความ (HTML/URL, whitespace, contractions), รวม `title` + `text` เป็นคอลัมน์ `review`, และกรอง duplicate/ข้อความสั้นเกินไปออก

In [10]:
cleaner = TextCleaner()
cleaned_data = cleaner.processing(raw_data)


Removed 0 duplicate reviews
Removed 0 empty/short reviews (< 3 chars)
Rows: 50000 -> 50000
Missing values per column:
sentiment         0
title             0
text              0
review            0
emphasis_count    0
dtype: int64
<class 'pandas.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype   
---  ------          --------------  -----   
 0   sentiment       50000 non-null  category
 1   title           50000 non-null  str     
 2   text            50000 non-null  str     
 3   review          50000 non-null  str     
 4   emphasis_count  50000 non-null  int64   
dtypes: category(1), int64(1), str(3)
memory usage: 48.6 MB


# 5. Data Quality Check & Results

ตรวจผลลัพธ์หลังทำความสะอาด — Validation Report (จำนวนแถวก่อน/หลัง, duplicate ที่ถูกลบ, Missing Values, Memory Usage) ถูก print ไว้แล้วในขั้นตอนก่อนหน้าโดย `processing()`; cell นี้ใช้ตรวจตัวอย่างผลลัพธ์แบบ Interactive เพิ่มเติม

In [11]:
cleaned_data.head()


,sentiment,title,text,review,emphasis_count
0,negative,expensive junk,this product consists of a piece of thin flexi...,expensive junk this product consists of a piec...,0
1,negative,toast too dark,"even on the lowest setting, the toast is too d...","toast too dark even on the lowest setting, the...",0
2,positive,excellent imagery...dumbed down story,i enjoyed this disc. the video is stunning. i ...,excellent imagery...dumbed down story i enjoye...,0
3,negative,are we pretending everyone is married?,the authors pretend that parents neither die n...,are we pretending everyone is married? the aut...,0
4,negative,not worth your time,"might as well just use a knife, this product h...",not worth your time might as well just use a k...,0


ตรวจว่ายังมีคำที่มี apostrophe contraction (เช่น `n't`) หลงเหลืออยู่ในคอลัมน์ `review` หลังผ่าน Cleaning Pipeline หรือไม่ — ใช้เป็น Diagnostic เพื่อดูช่องว่าง ของการ Normalize คำย่อ/คำสะกดผิดที่ยังต้องปรับปรุงต่อ

In [12]:
cleaned_data[cleaned_data['review'].str.contains("n't")]

,sentiment,title,text,review,emphasis_count
45,negative,item doen't work,this ups never worked from day one when i plug...,item doen't work this ups never worked from da...,0
1129,positive,hard to say if it is the 2nd or 3rd best,"if you really look at it, ""adventures on file ...",hard to say if it is the 2nd or 3rd best if yo...,0
1856,negative,too slow,i wanted to return it because the water is onl...,too slow i wanted to return it because the wat...,0
2853,negative,what a waste!!,a complete waste of money!! the earpieces did ...,what a waste!! a complete waste of money!! the...,0
3029,positive,incredible!,the author is very clear in his teachings. he ...,incredible! the author is very clear in his te...,0
...,...,...,...,...,...
47800,positive,this will send tingles down your spine.,this is quite possibly the best cd i have hear...,this will send tingles down your spine. this i...,0
47821,negative,not-so-sparkly,"my sister,lene, recieved this doll for her 6th...","not-so-sparkly my sister,lene, recieved this d...",0
48362,positive,brandon peters,this 28'typeii- 225 pound rated werner is a ni...,brandon peters this 28'typeii- 225 pound rated...,0
48978,negative,kind of a drag,this wan't the tracy i was hoping for. the who...,kind of a drag this wan't the tracy i was hopi...,0


# 6. Text Preprocessing (Tokenization + POS Filter + Lemmatization + Stopwords)

รวม tokenization, POS-based cleaning, lemmatization และ stopword removal เป็น spaCy pass เดียวผ่าน `TextProcessing.preprocess()` (`src/preprocessing.py`) — ใช้ `en_core_web_sm` (disable `parser`/`ner`, เก็บ `tagger`/`lemmatizer` ไว้เพราะต้องใช้ POS + lemma) กรอง token ที่ POS อยู่ใน `{'X', 'SPACE', 'PUNCT'}` และ stopword ออก แล้ว lemmatize + lower-case ให้เลย ได้คอลัมน์ `tokens` สุดท้ายพร้อมใช้งานต่อ

In [13]:
from preprocessing import TextProcessing

processor = TextProcessing()
processed_data = processor.preprocess(cleaned_data)
processed_data[['review', 'tokens']].head()

KeyboardInterrupt: 